In [ ]:
%%writefile test.cu
#include<stdio.h>
#include<stdlib.h>
#include<time.h>
#include"cuda_runtime.h"
#include "device_launch_parameters.h"

#define TILE_WIDTH 32

__global__ void MultMatGPU(float *p, float *m, float *n, int width);

int main() {
  int width = 1024; //1024x1024 행렬
  float *p = new float[width * width];
  float *m = new float[width * width];
  float *n = new float[width * width];

  for(int i = 0; i < width*width; ++i) {
    m[i] = rand() % 3 - 1.0;
    n[i] = rand() % 3 - 1.0; //m과 n의 각 인덱스에 값 할당
    //2차원 배열은 메모리상에선 1차원 배열을 끊어논 것에 불과해
    //이중 for문 없이도 구현할 수 있다.
    p[i] = 0.0;
  }

  cudaError_t cudaStatus = cudaSetDevice(0);
  float *dev_p, *dev_m, *dev_n;

  cudaStatus = cudaMalloc((void**)&dev_p, width*width * sizeof(float)); //m,n,p를 넣어줄 공간 확보
  cudaStatus = cudaMalloc((void**)&dev_m, width*width * sizeof(float));
  cudaStatus = cudaMalloc((void**)&dev_n, width*width * sizeof(float));

  cudaStatus = cudaMemcpy(dev_m, m, width*width * sizeof(float), cudaMemcpyHostToDevice); //m,n값 복사
  cudaStatus = cudaMemcpy(dev_n, n, width*width * sizeof(float), cudaMemcpyHostToDevice);

  dim3 dimGrid((width - 1) / TILE_WIDTH + 1, (width - 1) / TILE_WIDTH + 1);
  //2의 배수가 아닌 경우를 대비한 블록 개수 보정
  //비정방의 경우 해당 공식을 그대로 쓸 수 없음 변형 필수

  dim3 dimBlock(TILE_WIDTH, TILE_WIDTH);
  //thread는 32x32로 1024개 생성

  clock_t st = clock();
  MultMatGPU<<<dimGrid, dimBlock>>>(dev_p, dev_m, dev_n, width);
  cudaDeviceSynchronize();
  clock_t ed = clock();
  printf("Elapsed time = %u ms\n", ed - st);

  cudaStatus = cudaMemcpy(p, dev_p, width*width * sizeof(float), cudaMemcpyDeviceToHost);
  cudaStatus = cudaDeviceReset();

  delete[] p;
  delete[] m;
  delete[] n;
  cudaFree(dev_p);
  cudaFree(dev_m);
  cudaFree(dev_n);
}

__global__ void MultMatGPU(float *p, float *m, float *n, int width) {
  int i = blockIdx.y * blockDim.y + threadIdx.y;
  int j = blockIdx.x * blockDim.x + threadIdx.x; //row major (행우선)

  if(i < width && j < width) {
    float sum = 0.0;
    for(int k = 0; k < width; ++k) sum += m[i*width + k] * n[k*width+j];

    p[i*width+j] = sum;
  }
}

Overwriting test.cu


In [ ]:
!nvcc test.cu -o test
!./test

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
test.cu(67): warning #181-D: argument is incompatible with corresponding format string conversion (expected type "int" but argument has type "double")
      printf("p[%d][%d] = %d\n", i, j, p[i*width+j]);
                                       ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

test.cu: In function ‘int main()’:
test.cu:45:8: warning: format ‘%u’ expects argument of type ‘unsigned int’, but argument 2 has type ‘clock_t’ {aka ‘long int’} []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wformat=-Wformat=]8;;]
   45 |   printf("Elapsed time = %u ms\n", ed - st);
      |        ^~~~~~~~~~~~~~~~~~~~~~~~  ~~~~~~~
      |                                     |
      |                                     clock_t {aka long int}
p[1][0] = 0
p[1]

3x3 matirx multiplcation Example

In [ ]:
%%writefile test2.cu
#include<stdio.h>
#include<stdlib.h>
#include<time.h>
#include"cuda_runtime.h"
#include "device_launch_parameters.h"

#define TILE_WIDTH 32

__global__ void MultMatGPU(float *p, float *m, float *n, int width);

int main() {
  int width = 3; //3x3 행렬
  float *p = new float[width * width];

  float m[width*width] = {3.0, 1.0, 2.0, 4.0, 2.0, 1.0, 1.0, 2.0, 3.0};
  float n[width*width] = {2.0, 1.0, 3.0, 1.0, 2.0, 1.0, 0.0, 1.0, 2.0};

  cudaError_t cudaStatus = cudaSetDevice(0);
  float *dev_p, *dev_m, *dev_n;

  cudaStatus = cudaMalloc((void**)&dev_p, width*width * sizeof(float)); //m,n,p를 넣어줄 공간 확보
  cudaStatus = cudaMalloc((void**)&dev_m, width*width * sizeof(float));
  cudaStatus = cudaMalloc((void**)&dev_n, width*width * sizeof(float));

  cudaStatus = cudaMemcpy(dev_m, m, width*width * sizeof(float), cudaMemcpyHostToDevice); //m,n값 복사
  cudaStatus = cudaMemcpy(dev_n, n, width*width * sizeof(float), cudaMemcpyHostToDevice);

  dim3 dimGrid((width - 1) / TILE_WIDTH + 1, (width - 1) / TILE_WIDTH + 1);
  //2의 배수가 아닌 경우를 대비한 블록 개수 보정

  dim3 dimBlock(TILE_WIDTH, TILE_WIDTH);
  //thread는 32x32로 1024개 생성

  clock_t st = clock();
  MultMatGPU<<<dimGrid, dimBlock>>>(dev_p, dev_m, dev_n, width);
  cudaDeviceSynchronize();
  clock_t ed = clock();
  printf("Elapsed time = %u ms\n", ed - st);

  cudaStatus = cudaMemcpy(p, dev_p, width*width * sizeof(float), cudaMemcpyDeviceToHost);
  cudaStatus = cudaDeviceReset();

  delete[] p;
  cudaFree(dev_p);
  cudaFree(dev_m);
  cudaFree(dev_n);
}

__global__ void MultMatGPU(float *p, float *m, float *n, int width) {
  int i = blockIdx.y * blockDim.y + threadIdx.y;
  int j = blockIdx.x * blockDim.x + threadIdx.x; //row major (행우선)

  if(i < width && j < width) {
    float sum = 0.0;
    for(int k = 0; k < width; ++k)
      sum += m[i*width + k] * n[k*width+j];

    p[i * width + j] = sum;
    printf("p[%d][%d] = %f.0\n", i, j, p[i*width + j]);
  }
}

/*
p[1][0] = 10.000000.0
p[1][1] = 9.000000.0
p[1][2] = 16.000000.0
p[2][0] = 4.000000.0
p[2][1] = 8.000000.0
p[2][2] = 11.000000.0
p[0][0] = 7.000000.0
p[0][1] = 7.000000.0
p[0][2] = 14.000000.0
실행 결과는 다음과 같이 나오며 행에 맞춰 정리하면
7  7 14
10 9 16
4  8 11
로 정답에 맞게 출력됨을 볼 수 있다.

*/

Overwriting test2.cu


In [ ]:
!nvcc test2.cu -o test2
!./test2

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
test2.cu: In function ‘int main()’:
test2.cu:38:8: warning: format ‘%u’ expects argument of type ‘unsigned int’, but argument 2 has type ‘clock_t’ {aka ‘long int’} []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wformat=-Wformat=]8;;]
   38 |   printf("Elapsed time = %u ms\n", ed - st);
      |        ^~~~~~~~~~~~~~~~~~~~~~~~  ~~~~~~~
      |                                     |
      |                                     clock_t {aka long int}
p[1][0] = 10.000000.0
p[1][1] = 9.000000.0
p[1][2] = 16.000000.0
p[2][0] = 4.000000.0
p[2][1] = 8.000000.0
p[2][2] = 11.000000.0
p[0][0] = 7.000000.0
p[0][1] = 7.000000.0
p[0][2] = 14.000000.0
Elapsed time = 24024 ms
